# Atividade 2 — Indicadores: horas em cuidados e afazeres domésticos (mulheres x homens)

Base: **Tabela 1.1.1 (IBGE / PNAD Contínua 2022)** — número médio de horas semanais dedicadas a cuidados de pessoas e/ou afazeres domésticos, por sexo e cor ou raça.

A tabela `Tabela 1.1.1.xls` precisa estar na mesma pasta deste notebook.

In [ ]:
# python -m pip install pandas xlrd openpyxl
import pandas as pd

## 1. Abrir o arquivo e tratar o cabeçalho

In [ ]:
def ler_tabela_ibge(arquivo, nomes_colunas):
    """Lê as tabelas do IBGE pulando o cabeçalho de várias linhas.
    Pega da linha 'Brasil' até a linha 'Distrito Federal'."""
    bruto = pd.read_excel(arquivo, header=None)
    primeira_col = bruto.columns[0]
    inicio = bruto.index[bruto[primeira_col] == "Brasil"][0]
    fim = bruto.index[bruto[primeira_col] == "Distrito Federal"][0]
    df = bruto.loc[inicio:fim, bruto.columns[: len(nomes_colunas)]].copy()
    df.columns = nomes_colunas
    df = df.reset_index(drop=True)
    for c in nomes_colunas[1:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

REGIOES = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]

In [ ]:
colunas = [
    "uf_regiao",
    "total",
    "total_branca",
    "total_preta_parda",
    "homem_branca",
    "homem_preta_parda",
    "mulher_branca",
    "mulher_preta_parda",
]
indicador_1 = ler_tabela_ibge("Tabela 1.1.1.xls", colunas)
indicador_1.head(10)

In [ ]:
indicador_1.info()

## 2. Indicadores

In [ ]:
# Médias de homens e mulheres (média simples entre branca e preta/parda)
indicador_1["homens"] = (indicador_1["homem_branca"] + indicador_1["homem_preta_parda"]) / 2
indicador_1["mulheres"] = (indicador_1["mulher_branca"] + indicador_1["mulher_preta_parda"]) / 2

# Indicador 1: diferença de horas (mulheres - homens)
indicador_1["diferenca_horas"] = (indicador_1["mulheres"] - indicador_1["homens"]).round(1)

# Indicador 2: quantas vezes mais as mulheres trabalham em casa
indicador_1["razao_mulher_homem"] = (indicador_1["mulheres"] / indicador_1["homens"]).round(2)

# Indicador 3: diferença entre mulheres pretas/pardas e brancas
indicador_1["dif_mulher_preta_parda_x_branca"] = (
    indicador_1["mulher_preta_parda"] - indicador_1["mulher_branca"]
).round(1)

indicador_1[["uf_regiao", "homens", "mulheres", "diferenca_horas", "razao_mulher_homem",
             "dif_mulher_preta_parda_x_branca"]]

### Brasil e Grandes Regiões

In [ ]:
regioes = indicador_1[indicador_1["uf_regiao"].isin(["Brasil"] + REGIOES)]
regioes[["uf_regiao", "homens", "mulheres", "diferenca_horas", "razao_mulher_homem"]]

### Estados com maior e menor desigualdade

In [ ]:
estados = indicador_1[~indicador_1["uf_regiao"].isin(["Brasil"] + REGIOES)].copy()

print("Maior diferença (mulheres - homens):")
print(estados.sort_values("diferenca_horas", ascending=False)[["uf_regiao", "diferenca_horas"]].head(5))
print()
print("Menor diferença:")
print(estados.sort_values("diferenca_horas")[["uf_regiao", "diferenca_horas"]].head(5))

In [ ]:
mg = indicador_1.loc[indicador_1["uf_regiao"] == "Minas Gerais"].iloc[0]
print(f"Minas Gerais: mulheres {mg['mulheres']:.1f} h/semana x homens {mg['homens']:.1f} h/semana")
print(f"Diferença: {mg['diferenca_horas']} h  |  razão: {mg['razao_mulher_homem']}")

## Conclusão

Em todas as regiões e estados as mulheres dedicam bem mais horas por semana aos cuidados e afazeres domésticos que os homens. No Brasil, a média das mulheres fica perto de 21 horas contra cerca de 12 dos homens (quase o dobro). A diferença é maior no Nordeste e, entre as mulheres, as pretas ou pardas dedicam mais horas que as brancas em quase todos os estados.